In [68]:
import pandas as pd
import sqlite3
import json


In [94]:
orders = pd.read_csv("orders.csv")
print(orders.head())

   order_id  user_id  restaurant_id  order_date  total_amount  \
0         1     2508            450  18-02-2023        842.97   
1         2     2693            309  18-01-2023        546.68   
2         3     2084            107  15-07-2023        163.93   
3         4      319            224  04-10-2023       1155.97   
4         5     1064            293  25-12-2023       1321.91   

                  restaurant_name  
0               New Foods Chinese  
1  Ruchi Curry House Multicuisine  
2           Spice Kitchen Punjabi  
3          Darbar Kitchen Non-Veg  
4       Royal Eatery South Indian  


In [95]:
with open("users.json", "r") as f:
    users_data = json.load(f)

users = pd.DataFrame(users_data)
print(users.head())


   user_id    name       city membership
0        1  User_1    Chennai    Regular
1        2  User_2       Pune       Gold
2        3  User_3  Bangalore       Gold
3        4  User_4  Bangalore    Regular
4        5  User_5       Pune       Gold


In [97]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

with open("restaurants.sql", "r") as f:
    sql_script = f.read()

cursor.executescript(sql_script)


In [98]:
restaurants = pd.read_sql("SELECT * FROM restaurants", conn)

print(restaurants.head())


   restaurant_id restaurant_name  cuisine  rating
0              1    Restaurant_1  Chinese     4.8
1              2    Restaurant_2   Indian     4.1
2              3    Restaurant_3  Mexican     4.3
3              4    Restaurant_4  Chinese     4.1
4              5    Restaurant_5  Chinese     4.8


In [75]:
df = orders.merge(users, on="user_id", how="left") \
           .merge(restaurants, on="restaurant_id", how="left")

print(df.head())


   order_id  user_id  restaurant_id  order_date  total_amount  \
0         1     2508            450  18-02-2023        842.97   
1         2     2693            309  18-01-2023        546.68   
2         3     2084            107  15-07-2023        163.93   
3         4      319            224  04-10-2023       1155.97   
4         5     1064            293  25-12-2023       1321.91   

                restaurant_name_x       name       city membership  \
0               New Foods Chinese  User_2508  Hyderabad    Regular   
1  Ruchi Curry House Multicuisine  User_2693       Pune    Regular   
2           Spice Kitchen Punjabi  User_2084    Chennai       Gold   
3          Darbar Kitchen Non-Veg   User_319  Bangalore       Gold   
4       Royal Eatery South Indian  User_1064       Pune    Regular   

  restaurant_name_y  cuisine  rating  
0    Restaurant_450  Mexican     3.2  
1    Restaurant_309   Indian     4.5  
2    Restaurant_107  Mexican     4.0  
3    Restaurant_224  Chinese    

In [76]:
df["order_date"] = pd.to_datetime(df["order_date"], format="%d-%m-%Y")

In [77]:
print(df["order_date"].head())

0   2023-02-18
1   2023-01-18
2   2023-07-15
3   2023-10-04
4   2023-12-25
Name: order_date, dtype: datetime64[ns]


In [78]:
df.to_csv("final_food_delivery_dataset.csv", index=False)


In [79]:
df[df["membership"]=="Gold"].groupby("city")["total_amount"].sum()


,total_amount
city,
Bangalore,994702.59
Chennai,1080909.79
Hyderabad,896740.19
Pune,1003012.32


In [80]:
df.groupby("cuisine")["total_amount"].mean()

,total_amount
cuisine,
Chinese,798.389020
Indian,798.466011
Italian,799.448578
Mexican,808.021344


In [81]:
(df.groupby("user_id")["total_amount"].sum() > 1000).sum()

np.int64(2544)

In [82]:
bins = [3.0,3.5,4.0,4.5,5.0]
labels = ["3.0-3.5","3.6-4.0","4.1-4.5","4.6-5.0"]

df["rating_range"] = pd.cut(df["rating"], bins=bins, labels=labels)

df.groupby("rating_range")["total_amount"].sum()

/tmp/ipython-input-1192910983.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("rating_range")["total_amount"].sum()


,total_amount
rating_range,
3.0-3.5,1881754.57
3.6-4.0,1717494.41
4.1-4.5,1960326.26
4.6-5.0,2197030.75


In [83]:
df[df["membership"]=="Gold"].groupby("city")["total_amount"].mean()

,total_amount
city,
Bangalore,793.223756
Chennai,808.459080
Hyderabad,806.421034
Pune,781.162243


In [84]:
df.groupby("cuisine").agg(
    restaurants=("restaurant_id","nunique"),
    revenue=("total_amount","sum")
)

,restaurants,revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [85]:
df["membership"].value_counts(normalize=True) * 100

,proportion
membership,
Regular,50.13
Gold,49.87


In [87]:
df.groupby("restaurant_name_x").agg(
    avg_order=("total_amount", "mean"),
    total_orders=("order_id", "count")
).query("total_orders < 20").sort_values("avg_order", ascending=False).head()

,avg_order,total_orders
restaurant_name_x,,
Hotel Dhaba Multicuisine,1040.222308,13
Sri Mess Punjabi,1029.180833,12
Ruchi Biryani Punjabi,1002.140625,16
Sri Delights Pure Veg,989.467222,18
Classic Kitchen Family Restaurant,973.167895,19


In [88]:
df.groupby("restaurant_name_x").agg(
    avg_order=("total_amount", "mean"),
    total_orders=("order_id", "count")
).loc[
    ["Grand Cafe Punjabi",
     "Grand Restaurant South Indian",
     "Ruchi Mess Multicuisine",
     "Ruchi Foods Chinese"]
].query("total_orders < 20").sort_values("avg_order", ascending=False)


,avg_order,total_orders
restaurant_name_x,,
Ruchi Foods Chinese,686.603158,19


In [89]:
df.groupby(["membership","cuisine"])["total_amount"].sum()

membership  cuisine
Gold        Chinese     977713.74
            Indian      979312.31
            Italian    1005779.05
            Mexican    1012559.79
Regular     Chinese     952790.91
            Indian      992100.27
            Italian    1018424.75
            Mexican    1072943.30
Name: total_amount, dtype: float64

In [90]:
df["quarter"] = df["order_date"].dt.to_period("Q")

df.groupby("quarter")["total_amount"].sum()


,total_amount
quarter,
2023Q1,1993425.14
2023Q2,1945348.72
2023Q3,2037385.10
2023Q4,2018263.66
2024Q1,17201.50
